In [1]:
!pip install duckdb pandas

   ---------------------------------------- 0.0/13.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/13.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/13.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/13.7 MB ? eta -:--:--
    --------------------------------------- 0.3/13.7 MB ? eta -:--:--
    --------------------------------------- 0.3/13.7 MB ? eta -:--:--
   - -------------------------------------- 0.5/13.7 MB 605.9 kB/s eta 0:00:22
   - -------------------------------------- 0.5/13.7 MB 605.9 kB/s eta 0:00:22
   - -------------------------------------- 0.5/13.7 MB 605.9 kB/s eta 0:00:22
   -- ------------------------------------- 0.8/13.7 MB 523.8 kB/s eta 0:00:25
   --- ------------------------------------ 1.0/13.7 MB 567.1 kB/s eta 0:00:23
   --- ------------------------------------ 1.0/13.7 MB 567.1 kB/s eta 0:00:23
   --- ------------------------------------ 1.0/13.7 MB 567.1 kB/s eta 0:00:23
   --- --------------------

In [3]:
import duckdb
file_path = r"C:\Users\User\Documents\nigerian_retail_transactions_full.parquet"
duckdb.query(f"SELECT * FROM '{file_path}' LIMIT 5").df()

,transaction_id,account_id,customer_id,timestamp,amount_ngn,balance_before_ngn,balance_after_ngn,transaction_type,channel,merchant_category_code,merchant_name,location_lga,location_state,device_id,status,fraud_flag
0,42ae7cb0-052b-4c28-a149-8d7d2cc558a1,ACC-00000334,CUS-00000334,2024-09-14 14:59:00,2100.0,248561.187746,246461.187746,debit,web,4814,Airtel,Lagos LGA,Lagos,DEV-5b4184adc38945c2,failed,False
1,ce38456c-5bbb-4a83-992a-09e8fd59b49a,ACC-00004849,CUS-00004849,2024-12-26 01:54:00,73400.0,224711.795501,151311.795501,debit,web,5411,Grand Square,Benue LGA,Benue,DEV-b2b4b029db434254,success,False
2,9e69c19c-93c8-48fc-b2b8-e89d83b40936,ACC-00082189,CUS-00082189,2024-06-15 13:08:00,22600.0,114278.718868,136878.718868,credit,pos,7523,MegaPlaza Parking,Borno LGA,Borno,,success,False
3,eb707378-ac9a-4dd6-9a04-9cfe06970123,ACC-00010867,CUS-00010867,2023-11-18 18:45:00,45900.0,272525.157062,226625.157062,debit,mobile,,,Lagos LGA,Lagos,DEV-5f6ebeaeae504cdd,success,False
4,b29c3f2e-9712-4d9f-ab0e-32dc91493f3e,ACC-00004080,CUS-00004080,2024-04-21 22:48:00,80000.0,169907.403695,89907.403695,debit,branch,,,Lagos LGA,Lagos,,failed,False


In [4]:
row_count = duckdb.query(f"SELECT COUNT(*) FROM '{file_path}'").fetchone()[0]
print(f"Total Rows:{row_count:,}")
duckdb.query(f"DESCRIBE SELECT * FROM '{file_path}'").df()

Total Rows:5,000,000


,column_name,column_type,null,key,default,extra
0,transaction_id,VARCHAR,YES,None,None,None
1,account_id,VARCHAR,YES,None,None,None
2,customer_id,VARCHAR,YES,None,None,None
3,timestamp,TIMESTAMP_NS,YES,None,None,None
4,amount_ngn,DOUBLE,YES,None,None,None
5,balance_before_ngn,DOUBLE,YES,None,None,None
6,balance_after_ngn,DOUBLE,YES,None,None,None
7,transaction_type,VARCHAR,YES,None,None,None
8,channel,VARCHAR,YES,None,None,None
9,merchant_category_code,VARCHAR,YES,None,None,None


In [8]:
duckdb.query(f"""
SELECT transaction_id, COUNT(*) AS occurances
FROM '{file_path}'
GROUP BY transaction_id
HAVING COUNT(*) > 1
""").df()

,transaction_id,occurances


In [10]:
duckdb.query(f"""
SELECT channel, COUNT(*) AS total_count
FROM '{file_path}'
GROUP BY channel
ORDER BY total_count DESC
""").df()


,channel,total_count
0,mobile,1749043
1,pos,1501370
2,atm,999469
3,web,500493
4,branch,149372
5,ussd,75336
6,agent,24917


In [11]:
duckdb.query(f"""
SELECT transaction_type, COUNT(*) AS total_count
FROM '{file_path}'
GROUP BY transaction_type
ORDER BY total_count DESC
""").df()


,transaction_type,total_count
0,debit,3250541
1,credit,1749459


In [12]:
duckdb.query(f"""
SELECT location_lga, COUNT(*) AS total_count
FROM '{file_path}'
GROUP BY location_lga
ORDER BY total_count DESC
""").df()


,location_lga,total_count
0,Lagos LGA,1225836
1,Abuja (FCT) LGA,542759
2,Rivers LGA,446381
3,Kano LGA,288046
4,Anambra LGA,239518
5,Oyo LGA,238988
6,Imo LGA,186312
7,Kaduna LGA,159387
8,Delta LGA,135097
9,Edo LGA,125496


In [15]:
duckdb.query(f"""
SELECT
    COUNT(*) - COUNT(transaction_id) AS null_transaction_ids,
    COUNT(*) - COUNT(account_id) AS null_account_ids,
    COUNT(*) - COUNT(customer_id) AS null_customer_ids,
    COUNT(*) - COUNT(timestamp) AS null_timestamps,
    COUNT(*) - COUNT(amount_ngn) AS null_amounts,
    COUNT(*) - COUNT(transaction_type) AS null_transaction_types,
    COUNT(*) - COUNT(channel) AS null_channels,
    COUNT(*) - COUNT(merchant_category_code) AS null_mccs,
    COUNT(*) - COUNT(merchant_name) AS null_merchants,
    COUNT(*) - COUNT(location_lga) AS null_locations
FROM '{file_path}'
""").df()

,null_transaction_ids,null_account_ids,null_customer_ids,null_timestamps,null_amounts,null_transaction_types,null_channels,null_mccs,null_merchants,null_locations
0,0,0,0,0,0,0,0,0,0,0


In [17]:
duckdb.query(f"""
SELECT
    TRIM(transaction_id) AS transaction_id,
    TRIM(account_id) AS account_id,
    TRIM(customer_id) AS customer_id,
    timestamp,
    amount_ngn,
    balance_before_ngn,
    balance_after_ngn,
    TRIM(transaction_type) AS transaction_type,
    TRIM(channel) AS channel,
    TRIM(merchant_category_code) AS merchant_category_code,
    TRIM(merchant_name) AS merchant_name,
    TRIM(location_lga) AS location_lga
FROM '{file_path}'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,transaction_id,account_id,customer_id,timestamp,amount_ngn,balance_before_ngn,balance_after_ngn,transaction_type,channel,merchant_category_code,merchant_name,location_lga
0,42ae7cb0-052b-4c28-a149-8d7d2cc558a1,ACC-00000334,CUS-00000334,2024-09-14 14:59:00,2100.0,248561.187746,246461.187746,debit,web,4814,Airtel,Lagos LGA
1,ce38456c-5bbb-4a83-992a-09e8fd59b49a,ACC-00004849,CUS-00004849,2024-12-26 01:54:00,73400.0,224711.795501,151311.795501,debit,web,5411,Grand Square,Benue LGA
2,9e69c19c-93c8-48fc-b2b8-e89d83b40936,ACC-00082189,CUS-00082189,2024-06-15 13:08:00,22600.0,114278.718868,136878.718868,credit,pos,7523,MegaPlaza Parking,Borno LGA
3,eb707378-ac9a-4dd6-9a04-9cfe06970123,ACC-00010867,CUS-00010867,2023-11-18 18:45:00,45900.0,272525.157062,226625.157062,debit,mobile,,,Lagos LGA
4,b29c3f2e-9712-4d9f-ab0e-32dc91493f3e,ACC-00004080,CUS-00004080,2024-04-21 22:48:00,80000.0,169907.403695,89907.403695,debit,branch,,,Lagos LGA
...,...,...,...,...,...,...,...,...,...,...,...,...
4999995,c6f96081-96b2-4e90-822d-a24387f52d66,ACC-00001355,CUS-00001355,2023-12-02 20:59:00,45400.0,77658.204727,32258.204727,debit,mobile,,,Kano LGA
4999996,b25a9314-7825-4e07-adbf-afdc2ee6ca7c,ACC-00004405,CUS-00004405,2023-11-17 04:51:00,2200.0,84294.700157,82094.700157,debit,mobile,,,Gombe LGA
4999997,0f3b6f23-b490-479f-ad0f-b7b03d4a7cc0,ACC-00001626,CUS-00001626,2024-09-07 08:41:00,42100.0,312156.538456,270056.538456,debit,pos,4899,DSTV,Ogun LGA
4999998,922a73e0-ba3c-4d62-ba64-d9b7bc2d1378,ACC-00001335,CUS-00001335,2023-12-31 12:51:00,11100.0,171631.383138,160531.383138,debit,pos,4900,PHEDC,Kogi LGA


In [19]:
duckdb.query(f"""
SELECT
   transaction_id,
   account_id,
   customer_id,
   timestamp,
   ROUND(amount_ngn, 0) AS amount_ngn,
   ROUND(balance_before_ngn, 0) AS balance_before_ngn,
   ROUND(balance_after_ngn, 0) AS balance_after_ngn,
   transaction_type,
   channel,
   merchant_category_code,
   merchant_name,
   location_lga
FROM '{file_path}'
""").df()
   

,transaction_id,account_id,customer_id,timestamp,amount_ngn,balance_before_ngn,balance_after_ngn,transaction_type,channel,merchant_category_code,merchant_name,location_lga
0,42ae7cb0-052b-4c28-a149-8d7d2cc558a1,ACC-00000334,CUS-00000334,2024-09-14 14:59:00,2100.0,248561.0,246461.0,debit,web,4814,Airtel,Lagos LGA
1,ce38456c-5bbb-4a83-992a-09e8fd59b49a,ACC-00004849,CUS-00004849,2024-12-26 01:54:00,73400.0,224712.0,151312.0,debit,web,5411,Grand Square,Benue LGA
2,9e69c19c-93c8-48fc-b2b8-e89d83b40936,ACC-00082189,CUS-00082189,2024-06-15 13:08:00,22600.0,114279.0,136879.0,credit,pos,7523,MegaPlaza Parking,Borno LGA
3,eb707378-ac9a-4dd6-9a04-9cfe06970123,ACC-00010867,CUS-00010867,2023-11-18 18:45:00,45900.0,272525.0,226625.0,debit,mobile,,,Lagos LGA
4,b29c3f2e-9712-4d9f-ab0e-32dc91493f3e,ACC-00004080,CUS-00004080,2024-04-21 22:48:00,80000.0,169907.0,89907.0,debit,branch,,,Lagos LGA
...,...,...,...,...,...,...,...,...,...,...,...,...
4999995,c6f96081-96b2-4e90-822d-a24387f52d66,ACC-00001355,CUS-00001355,2023-12-02 20:59:00,45400.0,77658.0,32258.0,debit,mobile,,,Kano LGA
4999996,b25a9314-7825-4e07-adbf-afdc2ee6ca7c,ACC-00004405,CUS-00004405,2023-11-17 04:51:00,2200.0,84295.0,82095.0,debit,mobile,,,Gombe LGA
4999997,0f3b6f23-b490-479f-ad0f-b7b03d4a7cc0,ACC-00001626,CUS-00001626,2024-09-07 08:41:00,42100.0,312157.0,270057.0,debit,pos,4899,DSTV,Ogun LGA
4999998,922a73e0-ba3c-4d62-ba64-d9b7bc2d1378,ACC-00001335,CUS-00001335,2023-12-31 12:51:00,11100.0,171631.0,160531.0,debit,pos,4900,PHEDC,Kogi LGA


In [22]:
duckdb.query(f"""
SELECT
    transaction_id,
    account_id,
    customer_id,
    timestamp,
    CAST(amount_ngn AS BIGINT) AS amount_ngn,
    CAST(balance_before_ngn AS BIGINT) AS balance_before_ngn,
    CAST(balance_after_ngn AS BIGINT) AS balance_after_ngn,
    transaction_type,
    channel,
    merchant_category_code,
    merchant_name,
    location_lga
FROM '{file_path}'
""").df()
    

,transaction_id,account_id,customer_id,timestamp,amount_ngn,balance_before_ngn,balance_after_ngn,transaction_type,channel,merchant_category_code,merchant_name,location_lga
0,42ae7cb0-052b-4c28-a149-8d7d2cc558a1,ACC-00000334,CUS-00000334,2024-09-14 14:59:00,2100,248561,246461,debit,web,4814,Airtel,Lagos LGA
1,ce38456c-5bbb-4a83-992a-09e8fd59b49a,ACC-00004849,CUS-00004849,2024-12-26 01:54:00,73400,224712,151312,debit,web,5411,Grand Square,Benue LGA
2,9e69c19c-93c8-48fc-b2b8-e89d83b40936,ACC-00082189,CUS-00082189,2024-06-15 13:08:00,22600,114279,136879,credit,pos,7523,MegaPlaza Parking,Borno LGA
3,eb707378-ac9a-4dd6-9a04-9cfe06970123,ACC-00010867,CUS-00010867,2023-11-18 18:45:00,45900,272525,226625,debit,mobile,,,Lagos LGA
4,b29c3f2e-9712-4d9f-ab0e-32dc91493f3e,ACC-00004080,CUS-00004080,2024-04-21 22:48:00,80000,169907,89907,debit,branch,,,Lagos LGA
...,...,...,...,...,...,...,...,...,...,...,...,...
4999995,c6f96081-96b2-4e90-822d-a24387f52d66,ACC-00001355,CUS-00001355,2023-12-02 20:59:00,45400,77658,32258,debit,mobile,,,Kano LGA
4999996,b25a9314-7825-4e07-adbf-afdc2ee6ca7c,ACC-00004405,CUS-00004405,2023-11-17 04:51:00,2200,84295,82095,debit,mobile,,,Gombe LGA
4999997,0f3b6f23-b490-479f-ad0f-b7b03d4a7cc0,ACC-00001626,CUS-00001626,2024-09-07 08:41:00,42100,312157,270057,debit,pos,4899,DSTV,Ogun LGA
4999998,922a73e0-ba3c-4d62-ba64-d9b7bc2d1378,ACC-00001335,CUS-00001335,2023-12-31 12:51:00,11100,171631,160531,debit,pos,4900,PHEDC,Kogi LGA


In [26]:
duckdb.query(f"""
SELECT * REPLACE (
    CAST(amount_ngn AS BIGINT) AS amount_ngn,
    CAST(balance_before_ngn AS BIGINT) AS balance_before_ngn,
    CAST(balance_after_ngn AS BIGINT) AS balance_after_ngn,
    COALESCE(NULLIF(TRIM(merchant_category_code), ''),'N/A') AS merchant_category_code,
    COALESCE(NULLIF(TRIM(merchant_name), ''),  'Non-Merchant') AS merchant_name
)
FROM '{file_path}'
""").df()

,transaction_id,account_id,customer_id,timestamp,amount_ngn,balance_before_ngn,balance_after_ngn,transaction_type,channel,merchant_category_code,merchant_name,location_lga,location_state,device_id,status,fraud_flag
0,42ae7cb0-052b-4c28-a149-8d7d2cc558a1,ACC-00000334,CUS-00000334,2024-09-14 14:59:00,2100,248561,246461,debit,web,4814,Airtel,Lagos LGA,Lagos,DEV-5b4184adc38945c2,failed,False
1,ce38456c-5bbb-4a83-992a-09e8fd59b49a,ACC-00004849,CUS-00004849,2024-12-26 01:54:00,73400,224712,151312,debit,web,5411,Grand Square,Benue LGA,Benue,DEV-b2b4b029db434254,success,False
2,9e69c19c-93c8-48fc-b2b8-e89d83b40936,ACC-00082189,CUS-00082189,2024-06-15 13:08:00,22600,114279,136879,credit,pos,7523,MegaPlaza Parking,Borno LGA,Borno,,success,False
3,eb707378-ac9a-4dd6-9a04-9cfe06970123,ACC-00010867,CUS-00010867,2023-11-18 18:45:00,45900,272525,226625,debit,mobile,N/A,Non-Merchant,Lagos LGA,Lagos,DEV-5f6ebeaeae504cdd,success,False
4,b29c3f2e-9712-4d9f-ab0e-32dc91493f3e,ACC-00004080,CUS-00004080,2024-04-21 22:48:00,80000,169907,89907,debit,branch,N/A,Non-Merchant,Lagos LGA,Lagos,,failed,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4999995,c6f96081-96b2-4e90-822d-a24387f52d66,ACC-00001355,CUS-00001355,2023-12-02 20:59:00,45400,77658,32258,debit,mobile,N/A,Non-Merchant,Kano LGA,Kano,DEV-85c6b1ad42294bf4,success,False
4999996,b25a9314-7825-4e07-adbf-afdc2ee6ca7c,ACC-00004405,CUS-00004405,2023-11-17 04:51:00,2200,84295,82095,debit,mobile,N/A,Non-Merchant,Gombe LGA,Gombe,DEV-5c4f247f6ed6490d,success,False
4999997,0f3b6f23-b490-479f-ad0f-b7b03d4a7cc0,ACC-00001626,CUS-00001626,2024-09-07 08:41:00,42100,312157,270057,debit,pos,4899,DSTV,Ogun LGA,Ogun,,success,False
4999998,922a73e0-ba3c-4d62-ba64-d9b7bc2d1378,ACC-00001335,CUS-00001335,2023-12-31 12:51:00,11100,171631,160531,debit,pos,4900,PHEDC,Kogi LGA,Kogi,,success,False


In [15]:
import os
import duckdb

documents_dir = os.path.expanduser(r"~\Documents")
file_path = os.path.join(
    documents_dir, "nigerian_retail_transactions_full.parquet"
)
duckdb.query(
    f"""
    COPY (
      SELECT * REPLACE (
          CAST(amount_ngn AS BIGINT) AS amount_ngn,
          CAST(balance_before_ngn AS BIGINT) AS balance_before_ngn,
          CAST(balance_after_ngn AS BIGINT) AS balance_after_ngn,
          COALESCE(NULLIF(TRIM(merchant_category_code), ''), 'N/A') AS merchant_category_code,
          COALESCE(NULLIF(TRIM(merchant_name), ''), 'Non_merchant') AS merchant_name
        )
        FROM '{file_path}'
    ) TO 'cleaned_retail_transactions.csv' (HEADER, DELIMITER ',');
"""
)
print("Export complete! 'cleaned_retail_transaction.csv' is saved already for Tableau."
     )
    

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Export complete! 'cleaned_retail_transaction.csv' is saved already for Tableau.


In [16]:
import os
print(os.path.abspath("cleaned_retail_transactions.csv"))

C:\Users\User\cleaned_retail_transactions.csv


In [1]:
import os
print(os.getcwd())

C:\Users\User
